In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
from langchain_core.documents import Document
from typing import List
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size=chunk_size
        self.chunk_overlap=chunk_overlap
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "]

        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Load PDF

        loader=PyPDFLoader(pdf_path)
        pages=loader.load()

        ## Process each page

        processed_chunks=[]

        for page_num,page in enumerate(pages):
            ## clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())

        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")

        return text





In [3]:
preprocessor=SmartPDFProcessor()
preprocessor

In [4]:
## Process a PDF if available
try:
    smart_chunks=preprocessor.process_pdf("Amber25.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"  {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")

Processed into 3793 smart chunks

Sample chunk metadata:
  producer: pdfTeX-1.40.22
  creator: LaTeX with hyperref
  creationdate: 2025-07-28T09:42:00-06:00
  author: 
  title: 
  subject: 
  keywords: 
  moddate: 2025-07-28T09:42:00-06:00
  trapped: /False
  ptex.fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) kpathsea version 6.3.4/dev
  source: Amber25.pdf
  total_pages: 1061
  page: 1
  page_label: 1
  chunk_method: smart_pdf_processor
  char_count: 61


In [5]:
### Huggingface And OpenAI Models

from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
texts = [doc.page_content for doc in smart_chunks]
vectors = embeddings.embed_documents(texts)
metadata = [doc.metadata for doc in smart_chunks]
print(len(vectors), len(vectors[0]))

3793 384


### Warning: only run the Chroma.from_documents the first time or when you get a new PDF document.

In [7]:
# from langchain_community.vectorstores import Chroma
#
# persist_directory = "./chroma_db"
#
# vectorstore = Chroma.from_documents(
#     documents=smart_chunks,
#     embedding=embeddings,
#     persist_directory=persist_directory,
#     collection_name="rag_collection",
# )
#
# vectorstore.persist()  # depending on version; some auto-persist
# print(vectorstore._collection.count())

In [8]:
from langchain_community.vectorstores import Chroma

persist_directory = "./chroma_db"

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings,
    collection_name="rag_collection",
)

print(vectorstore._collection.count())

/tmp/ipykernel_370520/1394093444.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


3793


#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)
llm

ChatOllama(model='llama3.1:8b', temperature=0.0)

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [10]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# On hold: Do not include citations, references, or metadata.

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output the final answer.
For source citations, include source page number at the beginning.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4) If the Context contains relevant information, use it to answer as completely as possible.\n5) Do NOT mention an Persona, Identity, or Role in your answer.\n6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.\n\nSTYLE-\n- Start with a clear, d

In [12]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
     search_kwargs={"k":3} ## Retrieve top 3 relevant chunks
 )

retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x14e0e457eba0>, search_kwargs={'k': 3})

In [13]:
# ## Format the output documents for the prompt
# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)

In [14]:
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown_source")
        page = doc.metadata.get("page", "unknown_page")
        chunk = doc.metadata.get("chunk_id", i)
        formatted.append(
            f"[Source: {source} | Page: {page} | Chunk: {chunk}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [15]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x14e0e457eba0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4

In [16]:
response=rag_chain_lcel.invoke("What is Amber")
response

'Amber is the collective name for a suite of programs that allow users to carry out molecular dynamics simulations, particularly on biomolecules.\n\nTechnical Explanation:\nThe term "Amber" refers to both the software suite and the empirical force fields implemented within it. The code and force field are separate entities, with the force fields being in the public domain and the codes distributed under a license agreement.'

In [17]:
# Query using the LCEL approach
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    #docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
        print(doc.metadata)

In [18]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What can I use Amber for?")

Testing LCEL Chain:
Question: What can I use Amber for?
--------------------------------------------------
Answer: You can perform molecular dynamics simulations, particularly on biomolecules.

The Amber software suite is designed for carrying out molecular dynamics simulations, and it includes various programs that work together to provide a powerful framework for common calculations. The term "Amber" refers not only to the collective name of these programs but also to the empirical force fields implemented within them.

Source Documents:

--- Source 1 ---
Amber 2025 Reference Manual (Covers Amber24 and AmberTools25)...
{'page': 1, 'total_pages': 1061, 'keywords': '', 'trapped': '/False', 'creationdate': '2025-07-28T09:42:00-06:00', 'subject': '', 'chunk_method': 'smart_pdf_processor', 'moddate': '2025-07-28T09:42:00-06:00', 'title': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) kpathsea version 6.3.4/dev', 'producer': 'pdfTeX-1.40.

In [19]:
query_rag_lcel("What can you tell me about adaptively biased molecular dynamics?")

Question: What can you tell me about adaptively biased molecular dynamics?
--------------------------------------------------
Answer: **Adaptively Biased Molecular Dynamics (ABMD) is a method that belongs to the general category of umbrella sampling methods with a time-dependent potential.**

The ABMD method estimates the free energy of a reaction coordinate from an evolving ensemble of realizations, and uses that estimate to bias the system dynamics to flatten an effective free energy surface [Source: Amber25.pdf | Page: 559 | Chunk: 3]. This approach was developed by Prof. Celeste Sagui's group and implemented by Dr. V olodymyr Babin.

**Technical Explanation:** ABMD is a type of umbrella sampling method that uses a time-dependent potential to bias the system dynamics. This allows for the estimation of free energy surfaces in complex systems, which can be challenging with traditional molecular dynamics methods. The evolving ensemble of realizations enables the adaptation of the biasi

In [20]:
query_rag_lcel("What is QUICK?")

Question: What is QUICK?
--------------------------------------------------
Answer: QUICK is a GPU-enabled ab initio quantum chemistry program capable of performing electronic structure calculations on general organic/biomolecular systems.

Technical Explanation:
QUICK is an ab initio and density functional theory software that can perform efficient Hartree-Fock (HF) and density functional theory (DFT) energy and gradient calculations. It is available in four different types: serial, MPI parallel, CUDA serial, and CUDA MPI parallel, with respective executables quick, quick.MPI, quick.cuda, and quick.cuda.MPI.

Practical Guidance:
To run QUICK, users must source $AMBERHOME/amber.sh (or $AMBERHOME/amber.csh) before running the calculation. The input file should be specified as an argument to the executable, for example: `quick water.in` for a serial version or `mpirun -np 2 quick.cuda.MPI water.in` for a CUDA parallel version with 2 GPUs.

Source Documents:

--- Source 1 ---
Processing U